In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
    VotingRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [2]:
df = pd.read_csv("database/train_mod_tratado.csv")
df["Preco"] = pd.to_numeric(df["Preco"], errors="coerce")
df = df[df["Preco"].notna()].copy()

X = df.drop(columns=["Preco"])
y = df["Preco"]

df.shape


(9090, 25)

In [3]:
df['Preco'].isna().sum()

0

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

cat_cols = X_train.select_dtypes(include="object").columns.tolist()
num_cols = X_train.select_dtypes(exclude="object").columns.tolist()

print(cat_cols)
print(num_cols)


['Fabricante', 'Modelo', 'Categoria', 'Couro', 'Combustivel', 'Tipo_cambio', 'Tração', 'Portas', 'Cor', 'Data_ultima_lavagem', 'Adesivos_personalizados', 'Radio_AM_FM']
['Débitos', 'Ano', 'Volume_motor', 'Km', 'Cilindros', 'Airbags', 'Numero_proprietarios', 'Historico_troca_oleo', 'Idade', 'Km_por_ano', 'Dias_desde_ultima_lavagem', 'Km_imputado']


In [5]:
def criar_one_hot():
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=10)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore")

prep_modelos = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", criar_one_hot())
    ]), cat_cols),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols)
])

def regressao_log(modelo):
    return TransformedTargetRegressor(
        regressor=Pipeline([
            ("prep", prep_modelos),
            ("model", modelo)
        ]),
        func=np.log1p,
        inverse_func=np.expm1
    )


In [6]:
modelos = {
    "Ridge_log": {
        "pipeline": regressao_log(Ridge()),
        "params": {
            "regressor__model__alpha": [0.1, 1.0, 10.0, 100.0]
        }
    },
    "RandomForest_log": {
        "pipeline": regressao_log(RandomForestRegressor(random_state=42)),
        "params": {
            "regressor__model__n_estimators": [100, 200, 300, 500],
            "regressor__model__max_depth": [None, 10, 20, 30, 40],
            "regressor__model__min_samples_split": [2, 5, 10],
            "regressor__model__min_samples_leaf": [1, 2]
        }
    },
    "GradientBoosting_log": {
        "pipeline": regressao_log(GradientBoostingRegressor(random_state=42)),
        "params": {
            "regressor__model__n_estimators": [100, 200, 300, 500],
            "regressor__model__learning_rate": [0.05, 0.1, 0.01],
            "regressor__model__max_depth": [2, 3, 4, 5, 10]
        }
    },
    "ExtraTrees_log": {
        "pipeline": regressao_log(ExtraTreesRegressor(random_state=42)),
        "params": {
            "regressor__model__n_estimators": [100, 200, 300, 500],
            "regressor__model__max_depth": [None, 10, 20, 30, 40],
            "regressor__model__min_samples_split": [2, 5, 10],
            "regressor__model__min_samples_leaf": [1, 2]
        }
    },
    "LightGBM_L1_log": {
        "pipeline": regressao_log(LGBMRegressor(
            objective="regression_l1",
            metric="mae",
            random_state=42,
            n_jobs=1,
            verbosity=-1
        )),
        "params": {
            "regressor__model__n_estimators": [500, 700],
            "regressor__model__learning_rate": [0.025, 0.03],
            "regressor__model__num_leaves": [15, 31],
            "regressor__model__min_child_samples": [20, 30],
            "regressor__model__reg_lambda": [2.0, 5.0]
        }
    },
    "Blend_ExtraTrees_LightGBM_log": {
        "pipeline": regressao_log(VotingRegressor(
            estimators=[
                ("et", ExtraTreesRegressor(
                    n_estimators=400,
                    max_depth=20,
                    min_samples_leaf=1,
                    random_state=42,
                    n_jobs=1
                )),
                ("lgbm", LGBMRegressor(
                    objective="regression_l1",
                    metric="mae",
                    n_estimators=700,
                    learning_rate=0.025,
                    num_leaves=31,
                    min_child_samples=20,
                    reg_lambda=2.0,
                    random_state=42,
                    n_jobs=1,
                    verbosity=-1
                ))
            ],
            weights=[0.65, 0.35]
        )),
        "params": {}
    }
}

list(modelos.keys())


['Ridge_log',
 'RandomForest_log',
 'GradientBoosting_log',
 'ExtraTrees_log',
 'LightGBM_L1_log',
 'Blend_ExtraTrees_LightGBM_log']

In [7]:
resultados_metricas = []
melhores_modelos = {}

for nome_modelo, cfg in modelos.items():
    print(f"\nTreinando {nome_modelo}...")

    grid = GridSearchCV(
        estimator=cfg["pipeline"],
        param_grid=cfg["params"],
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    melhor_modelo = grid.best_estimator_
    y_pred = np.clip(melhor_modelo.predict(X_test), 0, None)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    resultados_metricas.append({
        "Modelo": nome_modelo,
        "Best_Params": str(grid.best_params_),
        "CV_MAE": -grid.best_score_,
        "MAE_teste": mae,
        "RMSE_teste": rmse,
        "R2_teste": r2
    })

    melhores_modelos[nome_modelo] = melhor_modelo



Treinando Ridge_log...

Treinando RandomForest_log...

Treinando GradientBoosting_log...

Treinando ExtraTrees_log...

Treinando LightGBM_L1_log...


c:\Users\pertile\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



Treinando Blend_ExtraTrees_LightGBM_log...


c:\Users\pertile\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [8]:
resultado_df = pd.DataFrame(resultados_metricas).sort_values("MAE_teste").reset_index(drop=True)
resultado_df


,Modelo,Best_Params,CV_MAE,MAE_teste,RMSE_teste,R2_teste
0,ExtraTrees_log,"{'regressor__model__max_depth': 40, 'regressor...",9882.097369,5988.262274,20894.218167,0.335724
1,Blend_ExtraTrees_LightGBM_log,{},9781.708165,6008.299774,20933.928575,0.333196
2,LightGBM_L1_log,"{'regressor__model__learning_rate': 0.03, 'reg...",9792.562458,6098.595482,20902.839195,0.335175
3,RandomForest_log,"{'regressor__model__max_depth': None, 'regress...",10071.716504,6205.115340,21198.309416,0.316247
4,GradientBoosting_log,"{'regressor__model__learning_rate': 0.05, 'reg...",10314.808561,6442.589427,21305.742383,0.309299
5,Ridge_log,{'regressor__model__alpha': 1.0},13775.011778,10041.055834,23930.726483,0.128619


In [9]:
os.makedirs("models", exist_ok=True)

for nome_modelo, modelo in melhores_modelos.items():
    nome_arquivo = nome_modelo.lower().replace(" ", "_") + ".pkl"
    joblib.dump(modelo, os.path.join("models", nome_arquivo))

melhor_nome = resultado_df.iloc[0]["Modelo"]
melhor_modelo_final = clone(melhores_modelos[melhor_nome])
melhor_modelo_final.fit(X, y)
joblib.dump(melhor_modelo_final, "models/best_price_model.pkl")

print("Melhor modelo:", melhor_nome)
print("Modelos salvos com sucesso.")


Melhor modelo: ExtraTrees_log
Modelos salvos com sucesso.


In [10]:
melhor_nome = resultado_df.iloc[0]["Modelo"]
y_pred_best = np.clip(melhores_modelos[melhor_nome].predict(X_test), 0, None)

comparacao = pd.DataFrame({
    "Preco_real": y_test.values,
    "Preco_predito": y_pred_best,
    "Erro": y_pred_best - y_test.values,
    "Erro_abs": np.abs(y_pred_best - y_test.values)
}, index=y_test.index).sort_values("Erro_abs", ascending=False)

comparacao.head(20)


,Preco_real,Preco_predito,Erro,Erro_abs
9088,866800.0,14155.418601,-852644.581399,852644.581399
9085,297930.0,65400.374270,-232529.625730,232529.625730
9086,297930.0,87768.417396,-210161.582604,210161.582604
9083,250574.0,108823.414553,-141750.585447,141750.585447
9079,156805.0,22225.251967,-134579.748033,134579.748033
9053,114468.0,5936.956005,-108531.043995,108531.043995
9065,130148.0,30665.702183,-99482.297817,99482.297817
9030,98787.0,16649.055520,-82137.944480,82137.944480
9035,100355.0,18955.836439,-81399.163561,81399.163561
8997,88595.0,10582.886401,-78012.113599,78012.113599


In [11]:
comparacao.head(100)

,Preco_real,Preco_predito,Erro,Erro_abs
9088,866800.0,14155.418601,-852644.581399,852644.581399
9085,297930.0,65400.374270,-232529.625730,232529.625730
9086,297930.0,87768.417396,-210161.582604,210161.582604
9083,250574.0,108823.414553,-141750.585447,141750.585447
9079,156805.0,22225.251967,-134579.748033,134579.748033
...,...,...,...,...
3127,8000.0,34855.373634,26855.373634,26855.373634
8994,87927.0,61125.771058,-26801.228942,26801.228942
7607,29793.0,3010.846786,-26782.153214,26782.153214
8890,67000.0,40269.311779,-26730.688221,26730.688221
